# Step 1: Extract, Clean, Chunk, and Convert Course Materials

This notebook prepares the knowledge base for the Course Q&A RAG system. It processes three course books and produces the files used by `02_rag.ipynb` and the Streamlit app.

| Course | Source PDF | Output format |
|---|---|---|
| Multimodal Deep Learning | `multimodal_dl/mdl.pdf` | `.docx` |
| Python 3 Tutorial | `python/Python - 3+ Version Tutorial.pdf` | `.txt` |
| Computer Vision (Szeliski) | `computer_vision/SzeliskiBookDraft_20210930.pdf` | `.csv` |

**Steps:**
1. Extract text from each PDF (PyMuPDF)
2. Clean it: remove headers, page numbers, bibliographies, tables of contents, and noisy pages
3. Split it into chunks with section and page metadata
4. Save the chunks as `.jsonl`, then convert them to `.txt`, `.docx`, and `.csv`
5. Verify that each format loads back to the same chunks (round-trip test)

**Note:** the source PDFs are not included in the repository. To re-run the extraction, place them in the folders listed above. The converted files in `data/raw_formats/` are enough to run the app.

## 1. Multimodal Deep Learning

Extract the pages, drop near-empty pages, strip running headers and page numbers, cut the bibliography, and split into ~1200-character chunks tagged with section and printed page numbers.

In [1]:
import fitz, glob

path = glob.glob("multimodal_dl/*.pdf")[0]
doc = fitz.open(path)
print(path, "| pages:", len(doc))

empty = [i + 1 for i, p in enumerate(doc) if len(p.get_text().strip()) < 100]
print("near-empty pages:", len(empty), empty[:20])
print(doc[20].get_text()[:800])

C:\Users\ramez\AppData\Roaming\Python\Python314\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


multimodal_dl\mdl.pdf | pages: 272
near-empty pages: 9 [1, 2, 4, 12, 45, 88, 186, 240, 246]
2.1 State-of-the-art in NLP
15
type of input and output, a design with two main parts could be useful.
The ﬁrst one is called the encoder because, in this part of the network, a
variable length input sequence is transformed into a ﬁxed state. Next, the
second component called the decoder maps the encoded state to an output of
a variable length sequence. As a whole, it is known as an encoder-decoder or
sequence-to-sequence architecture and has become an eﬀective and standard
approach for many applications which even recurrent neural networks with
gated hidden units have trouble solving successfully. Deep RNNs may have
a chance, but diﬀerent architectures like encoder-decoder have proven to be
the most eﬀective. It can even deal with diﬀerent word orders and active,
as well as passive voice 


In [2]:
import re, unicodedata

def clean_page(raw):
    text = unicodedata.normalize("NFKC", raw)
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    header, printed = None, None
    for l in lines[:2]:
        if l.isdigit():
            printed = int(l)
        elif header is None and len(l) < 70 and not l.endswith((".", ",")):
            header = l
    skip = sum(1 for l in lines[:2] if l.isdigit() or l == header)
    body = re.sub(r"\s+", " ", " ".join(lines[skip:]))
    if len(body) < 200:
        return None
    return {"header": header, "printed_page": printed, "text": body}

pages = []
for i, p in enumerate(doc, start=1):
    c = clean_page(p.get_text())
    if c:
        pages.append({"pdf_page": i, **c})

print("kept:", len(pages), "of", len(doc))
p = pages[20]
print(p["pdf_page"], "|", p["header"], "|", p["printed_page"])
print(p["text"][:500])

kept: 259 of 272
27 | 2.1 State-of-the-art in NLP | 21
FIGURE 2.11: Sequential processing of recurrent model (Source: Manning et al. (2022)). data sets. indicates the minimum number of steps before the respective state can be calculated. FIGURE 2.12: Sequential processing of recurrent model with number of steps indicated (Source: Manning et al. (2022)). After proving that attention dramatically increases performance, google researchers took it further and based transformers solely on attention, so without any RNNs. For this reason, the paper in whic


In [3]:
import re

OFFSET = 6  # printed page = pdf page - 6

first_bib = next(p["pdf_page"] for p in pages if "Bibliography" in (p["header"] or ""))
print("bibliography starts at pdf page:", first_bib)

content, last_section = [], None
for p in pages:
    if not (9 <= p["pdf_page"] < first_bib) or p["header"] == "Acknowledgements":
        continue
    h = p["header"] or ""
    ch = re.match(r"^(\d+)\s", h)
    if re.match(r"^\d+\.\d+\s", h):
        last_section = h
    elif ch and last_section and last_section.startswith(ch.group(1) + "."):
        h = last_section
    content.append({**p, "printed_page": p["pdf_page"] - OFFSET, "section": h})

print("kept:", len(content))
for p in content[:3] + content[-3:]:
    print(p["pdf_page"], "|", p["section"], "|", p["printed_page"], "|", p["text"][:70])

bibliography starts at pdf page: 247
kept: 229
9 | Introduction | 3 | Author: Nadja Sauter Supervisor: Matthias Aßenmacher 1.1 Introduction 
10 | 1 Introduction | 4 | architectures have been developed in recent years. This book gives an 
11 | 1.2 Outline of the Booklet | 5 | supporting CV models like CLIP (Radford et al., 2021b), ALIGN (Jia et 
242 | 5 Conclusion | 236 | (Brown et al., 2020). Increasing the number of parameters often helps 
243 | Epilogue | 237 | Author: Matthias Aßenmacher Since this project was realized in a limit
244 | 6 Epilogue | 238 | 6.2 Creating videos Also more recently, research has focussed on not o


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from itertools import groupby
import bisect

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200, chunk_overlap=200, add_start_index=True
)

chunks = []
for section, grp in groupby(content, key=lambda p: p["section"]):
    grp = list(grp)
    text, starts = "", []
    for p in grp:
        starts.append(len(text))
        text += p["text"] + " "
    for d in splitter.create_documents([text]):
        s = d.metadata["start_index"]
        e = s + len(d.page_content) - 1
        pa = grp[bisect.bisect_right(starts, s) - 1]["printed_page"]
        pb = grp[bisect.bisect_right(starts, e) - 1]["printed_page"]
        chunks.append({
            "course": "Multimodal Deep Learning",
            "source": "mdl.pdf",
            "section": section,
            "pages": f"{pa}" if pa == pb else f"{pa}-{pb}",
            "text": d.page_content,
        })

lens = [len(c["text"]) for c in chunks]
print("chunks:", len(chunks), "| min/avg/max:", min(lens), sum(lens) // len(lens), max(lens))
print("tiny (<150):", sum(l < 150 for l in lens))
c = chunks[40]
print(c["section"], "| p.", c["pages"])
print(c["text"][:400])

chunks: 510 | min/avg/max: 213 1162 1200
tiny (<150): 0
2.1 State-of-the-art in NLP | p. 21
in which they were introduced is called "Attention is all you need". Spoiler: It is not quite all we need, but more about that on the following pages. Transformers have achieved great results on multiple settings such as machine translation and document generation. Their parallelizability allows for efficient pretraining and leads them to be the standard model architecture. In fact, all top models


In [5]:
import json, os

os.makedirs("data/processed", exist_ok=True)
out = "data/processed/mdl_chunks.jsonl"
with open(out, "w", encoding="utf-8") as f:
    for i, c in enumerate(chunks):
        f.write(json.dumps({"id": f"mdl-{i:04d}", **c}, ensure_ascii=False) + "\n")

print(os.path.getsize(out) // 1024, "KB |", len(chunks), "chunks")

658 KB | 510 chunks


## 2. Python 3 Tutorial

Detect chapter boundaries, keep chapters 1-20, remove the `os.*` reference section, and split into ~1500-character chunks. Each method reference (for example `String split() Method`) becomes its own section so it can be retrieved precisely.

In [6]:
files = glob.glob("python/*.pdf")
print(files)
path = next((f for f in files if "tutorial" in f.lower()), files[0])
pdoc = fitz.open(path)
print(path, "| pages:", len(pdoc))

chap = re.compile(r"^\s*(\d+)\.\s*Python 3\s*[–-]\s*(.+?)\s*$", re.M)
for i, p in enumerate(pdoc, start=1):
    for m in chap.finditer(p.get_text()):
        print(i, "|", m.group(1), "|", m.group(2))

print("-----")
print(pdoc[60].get_text()[:600])

['python\\Python - 3+ Version Tutorial.pdf']
python\Python - 3+ Version Tutorial.pdf | pages: 512
3 | 1 | What is New? .......................................................................................................................... 2
3 | 2 | Overview ................................................................................................................................. 6
3 | 3 | Environment Setup.................................................................................................................. 8
3 | 4 | Basic Syntax .......................................................................................................................... 13
4 | 5 | Variable Types ....................................................................................................................... 23
4 | 6 | Basic Operators ..................................................................................................................... 30
4 | 7 | Decision Making ...

In [7]:
OFFSET = 13  # printed page = pdf page - 13
chap_re = re.compile(r"^\s*(\d+)\.\s*Python 3\s*[–-]\s*(.+?)\s*$", re.M)

def find_chapter(body):
    for m in chap_re.finditer(body):
        title = re.sub(r"\s+", " ", m.group(2))
        if "..." not in title:  # skip table-of-contents lines
            return m, int(m.group(1)), title
    return None

def clean_py_page(raw):
    text = unicodedata.normalize("NFKC", raw)
    text = re.sub(r"[\ue000-\uf8ff]", "- ", text)  # bullet glyphs
    lines = [l.rstrip() for l in text.split("\n")]
    while lines and not lines[0].strip():
        lines.pop(0)
    if lines and lines[0].strip() == "Python 3":
        lines.pop(0)
        if lines and lines[0].strip().isdigit():
            lines.pop(0)
    body = "\n".join(lines)
    chapter = None
    found = find_chapter(body)
    if found:
        m, num, title = found
        chapter = (num, title)
        body = body[:m.start()] + body[m.end():]
    return re.sub(r"\n{3,}", "\n\n", body).strip(), chapter

py_pages, chapters, current = [], [], None
for i, p in enumerate(pdoc, start=1):
    if i < 15:
        continue
    body, ch = clean_py_page(p.get_text())
    if ch:
        if ch[0] > 20:
            break
        current = f"{ch[0]}. {ch[1]}"
        chapters.append((ch[0], ch[1], i, i - OFFSET))
    if current and len(body) >= 80:
        py_pages.append({"pdf_page": i, "printed_page": i - OFFSET,
                         "section": current, "text": body})

print("pages kept:", len(py_pages))
if py_pages:
    print("pdf", py_pages[0]["pdf_page"], "-", py_pages[-1]["pdf_page"])
for n, t, pdf, bk in chapters:
    print(n, "|", t, "| pdf", pdf, "| book", bk)
    
s = next((p for p in py_pages if p["section"].startswith("2.")), None)
if s:
    print("-----")
    print(s["text"][:350])

pages kept: 352
pdf 15 - 370
1 | What is New? | pdf 15 | book 2
2 | Overview | pdf 19 | book 6
3 | Environment Setup | pdf 21 | book 8
4 | Basic Syntax | pdf 26 | book 13
5 | Variable Types | pdf 36 | book 23
6 | Basic Operators | pdf 43 | book 30
7 | Decision Making | pdf 56 | book 43
8 | Loops | pdf 64 | book 51
9 | Numbers | pdf 81 | book 68
10 | Strings | pdf 112 | book 99
11 | Lists | pdf 156 | book 143
12 | Tuples | pdf 170 | book 157
13 | Dictionary | pdf 177 | book 164
14 | Date & Time | pdf 191 | book 178
15 | Functions | pdf 212 | book 199
16 | Modules | pdf 221 | book 208
17 | Files I/O | pdf 228 | book 215
18 | Exceptions Handling | pdf 336 | book 323
19 | Object Oriented | pdf 347 | book 334
20 | Regular Expressions | pdf 359 | book 346
-----
Python is a high-level, interpreted, interactive and object-oriented scripting language.
Python is designed to be highly readable. It uses English keywords frequently whereas the
other languages use punctuations. It has fewer syntacti

In [8]:
# 1) fix bullets that ended up on their own line
for p in py_pages:
    p["text"] = re.sub(r"(?m)^- *\n", "- ", p["text"])

# 2) cut the os.* methods section (end of chapter 17)
marker = "OS File/Directory Methods"
cut = next(i for i, p in enumerate(py_pages) if marker in p["text"])
py_pages[cut]["text"] = py_pages[cut]["text"].split(marker)[0].strip()
ch18 = next(i for i, p in enumerate(py_pages) if p["section"].startswith("18."))
py_pages = py_pages[:cut + 1] + py_pages[ch18:]

print("pages kept:", len(py_pages), "| cut at pdf page:", py_pages[cut]["pdf_page"])
s = next(p for p in py_pages if p["section"].startswith("2."))
print(s["text"][:300])

pages kept: 272 | cut at pdf page: 254
Python is a high-level, interpreted, interactive and object-oriented scripting language.
Python is designed to be highly readable. It uses English keywords frequently whereas the
other languages use punctuations. It has fewer syntactical constructions than other
languages.
- Python is Interpreted: P


In [9]:
head_re = re.compile(
    r"(?m)^(?:Number|String|List|Tuple|Dictionary|Time|File)\s*[\w.]+\(\)\s*Method\s*$"
)
py_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, chunk_overlap=150, add_start_index=True,
    separators=["\n\n", "\n", " ", ""],
)

py_chunks = []
for chapter, grp in groupby(py_pages, key=lambda p: p["section"]):
    grp = list(grp)
    text, starts = "", []
    for p in grp:
        starts.append(len(text))
        text += p["text"] + "\n"
    cuts = [m.start() for m in head_re.finditer(text)]
    bounds = sorted(set([0] + cuts)) + [len(text)]
    for a, b in zip(bounds, bounds[1:]):
        seg = text[a:b]
        if not seg.strip():
            continue
        first = seg.strip().split("\n")[0].strip()
        section = f"{chapter} > {first}" if a in cuts else chapter
        for d in py_splitter.create_documents([seg]):
            s = a + d.metadata["start_index"]
            e = s + len(d.page_content) - 1
            pa = grp[bisect.bisect_right(starts, s) - 1]["printed_page"]
            pb = grp[bisect.bisect_right(starts, e) - 1]["printed_page"]
            py_chunks.append({
                "course": "Python 3 Tutorial",
                "source": "Python - 3+ Version Tutorial.pdf",
                "section": section,
                "pages": f"{pa}" if pa == pb else f"{pa}-{pb}",
                "text": d.page_content,
            })

lens = [len(c["text"]) for c in py_chunks]
print("chunks:", len(py_chunks), "| min/avg/max:", min(lens), sum(lens) // len(lens), max(lens))
print("tiny (<150):", sum(l < 150 for l in lens))
print("method chunks:", sum(">" in c["section"] for c in py_chunks))
c = next(c for c in py_chunks if "String split() Method" in c["section"])
print("-----")
print(c["section"], "| p.", c["pages"])
print(c["text"][:500])

chunks: 314 | min/avg/max: 40 953 1500
tiny (<150): 7
method chunks: 135
-----
10. Strings > String split() Method | p. 134
String split() Method
Description
The split() method returns a list of all the words in the string, using str as the separator
(splits on all whitespace if left unspecified), optionally limiting the number of splits to num.
Syntax
Following is the syntax for split() method-
str.split(str="", num=string.count(str)).
Parameters
- str - This is any delimeter, by default it is space.

- num - this is number of lines to be made
Return Value
This method returns a list of lines.
Example
The following e


In [10]:
filler = ("Let us go through",)
merged = []
for c in py_chunks:
    if len(c["text"]) < 150:
        if c["text"].startswith(filler):
            continue
        prev = merged[-1] if merged else None
        if prev and prev["section"] == c["section"] and len(prev["text"]) + len(c["text"]) <= 1800:
            prev["text"] += "\n" + c["text"]
            if c["pages"] != prev["pages"]:
                prev["pages"] = f'{prev["pages"].split("-")[0]}-{c["pages"].split("-")[-1]}'
            continue
    merged.append(dict(c))

py_chunks = merged
lens = [len(c["text"]) for c in py_chunks]
print("chunks:", len(py_chunks), "| tiny left:", sum(l < 150 for l in lens))

out = "data/processed/python_chunks.jsonl"
with open(out, "w", encoding="utf-8") as f:
    for i, c in enumerate(py_chunks):
        f.write(json.dumps({"id": f"py-{i:04d}", **c}, ensure_ascii=False) + "\n")
print(os.path.getsize(out) // 1024, "KB saved")

chunks: 307 | tiny left: 0
351 KB saved


## 3. Computer Vision (Szeliski)

Keep chapters 2, 3, 5, 6, and 7. Remove table-of-contents pages, figure panel labels, and exercise sections, repair hyphenated words, drop math-heavy chunks, and split into ~1200-character chunks.

In [11]:
files = glob.glob("computer_vision/*.pdf")
print(files)
sdoc = fitz.open(files[0])
print("pages:", len(sdoc))

for i, p in enumerate(sdoc, start=1):
    m = re.match(r"\s*Chapter\s+(\d+)\b", p.get_text()[:80])
    if m:
        print("chapter", m.group(1), "| pdf page", i)

for n in (300, 301):
    t = [l for l in sdoc[n - 1].get_text().strip().split("\n") if l.strip()]
    print(n, "| FIRST:", t[:3], "| LAST:", t[-3:])

['computer_vision\\SzeliskiBookDraft_20210930.pdf']
pages: 1232
chapter 1 | pdf page 27
chapter 2 | pdf page 59
chapter 3 | pdf page 133
chapter 4 | pdf page 217
chapter 5 | pdf page 261
chapter 6 | pdf page 369
chapter 7 | pdf page 443
chapter 8 | pdf page 527
chapter 9 | pdf page 581
chapter 10 | pdf page 633
chapter 11 | pdf page 707
chapter 12 | pdf page 775
chapter 13 | pdf page 831
chapter 14 | pdf page 887
chapter 15 | pdf page 941
300 | FIRST: ['274', 'Computer Vision: Algorithms and Applications (September 30, 2021 draft)', '(a)'] | LAST: ['As we saw in Section 4.1.1, quadratic or p-norm penalties on the weights (4.9) can be used', 'to improve the conditioning of the system and to reduce overﬁtting. Setting p = 2 results in', 'the usual L2 regularization and makes large weights smaller, whereas using p = 1 is called']
301 | FIRST: ['5.3 Deep neural networks', '275', 'Figure 5.28'] | LAST: ['around, and therefore introduce much larger changes in the input vector space, while st

In [12]:
OFFSET = 26  # book page = pdf page - 26
starts = {2: 59, 3: 133, 4: 217, 5: 261, 6: 369, 7: 443, 8: 527}
KEEP = [2, 3, 5, 6, 7]
sec_re = re.compile(r"^(\d+\.\d+(?:\.\d+)?)\s+([A-Z0-9][^.,;]{2,70})$")
head_re = re.compile(r"^\d+\.\d+\s+[A-Z0-9]")
end_re = re.compile(r"^\d+\.\d+\s+(?:Additional reading|Exercises)\s*$")
panel_re = re.compile(r"^\(?[a-zA-Z]\)$")

def merge_numbers(lines):
    out, k = [], 0
    while k < len(lines):
        l = lines[k]
        if ". . ." in l:
            k += 1
            continue
        if (re.fullmatch(r"[2-8]\.\d+(?:\.\d+)?", l) and k + 1 < len(lines)
                and re.match(r"[A-Z][^.;]{2,70}$", lines[k + 1])):
            out.append(l + " " + lines[k + 1])
            k += 2
        else:
            out.append(l)
            k += 1
    return out

def clean_cv_page(raw):
    raw_lines = [l.strip() for l in unicodedata.normalize("NFKC", raw).split("\n") if l.strip()]
    if sum(". . ." in l for l in raw_lines) >= 3:   # chapter-opening TOC page
        return [], False, None
    lines = merge_numbers(raw_lines)
    done = False
    for k, l in enumerate(lines):
        if end_re.match(l):
            lines, done = lines[:k], True
            break
    out, hdr = [], None
    for k, l in enumerate(lines):
        if k < 4 and (l.isdigit() or l.startswith("Computer Vision: Algorithms")):
            continue
        if k < 4 and hdr is None and head_re.match(l):
            hdr = l
            continue
        if panel_re.match(l):
            continue
        out.append(l)
    return out, done, hdr

def seg(i, section, buf):
    return {"pdf_page": i, "printed_page": i - OFFSET, "section": section,
            "text": re.sub(r"\s+", " ", " ".join(buf)).strip()}

cv_segs = []
for ch in KEEP:
    section, done = f"Chapter {ch}", False
    for i in range(starts[ch], starts[ch + 1]):
        if done:
            break
        lines, done, hdr = clean_cv_page(sdoc[i - 1].get_text())
        if hdr and hdr.startswith(f"{ch}.") and not section.split()[0].startswith(hdr.split()[0]):
            section = hdr
        buf = []
        for l in lines:
            s = sec_re.match(l)
            if s and s.group(1).startswith(f"{ch}."):
                if buf:
                    cv_segs.append(seg(i, section, buf))
                buf, section = [], l
            buf.append(l)
        if buf:
            cv_segs.append(seg(i, section, buf))

cv_segs = [s for s in cv_segs if len(s["text"]) >= 60]
print("segments:", len(cv_segs))
for ch in KEEP:
    ps = [s["printed_page"] for s in cv_segs if s["section"].startswith((f"{ch}.", f"Chapter {ch}"))]
    print("chapter", ch, "| book pages", (min(ps), max(ps)) if ps else "NONE")
secs = list(dict.fromkeys(s["section"] for s in cv_segs))
print("sections:", len(secs))
print(secs[:6], secs[-3:])
x = next((s for s in cv_segs if s["section"].startswith("5.3.1")), None)
print("-----")
if x:
    print(x["section"], "| p.", x["printed_page"])
    print(x["text"][:350])

segments: 480
chapter 2 | book pages (34, 100)
chapter 3 | book pages (108, 178)
chapter 5 | book pages (236, 335)
chapter 6 | book pages (344, 408)
chapter 7 | book pages (418, 490)
sections: 114
['Chapter 2', '2.1 Geometric primitives and transformations', '2.1.5 Lens distortions', '2.2 Photometric image formation', '2.2.1 Lighting', '2.2.2 Reflectance and shading'] ['7.5.1 Graph-based segmentation', '7.5.2 Mean shift', '7.5.3 Normalized cuts']
-----
5.3.1 Weights and layers | p. 270
5.3.1 Weights and layers Deep neural networks (DNNs) are feedforward computation graphs composed of thousands of simple interconnected “neurons” (units), which, much like logistic regression (5.18), per- form weighted sums of their inputs si = wT i xi + bi (5.45) followed by a non-linear activation function re-mapping, yi = h(si), (5.46) as illustr


In [13]:
def fix_text(t):
    # "per- form" -> "perform" (keep "low- and high-pass")
    return re.sub(r"([a-z])- (?!and\b|or\b)([a-z])", r"\1\2", t)

cv_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200, chunk_overlap=200, add_start_index=True
)

cv_chunks = []
for section, grp in groupby(cv_segs, key=lambda s: s["section"]):
    grp = list(grp)
    text, starts = "", []
    for s in grp:
        starts.append(len(text))
        text += fix_text(s["text"]) + " "
    for d in cv_splitter.create_documents([text]):
        a = d.metadata["start_index"]
        b = a + len(d.page_content) - 1
        pa = grp[bisect.bisect_right(starts, a) - 1]["printed_page"]
        pb = grp[bisect.bisect_right(starts, b) - 1]["printed_page"]
        cv_chunks.append({
            "course": "Computer Vision (Szeliski)",
            "source": "SzeliskiBookDraft_20210930.pdf",
            "section": section,
            "pages": f"{pa}" if pa == pb else f"{pa}-{pb}",
            "text": d.page_content,
        })

before = len(cv_chunks)
cv_chunks = [c for c in cv_chunks if len(c["text"]) >= 150]
lens = [len(c["text"]) for c in cv_chunks]
alpha = [sum(ch.isalpha() for ch in c["text"]) / len(c["text"]) for c in cv_chunks]
print("chunks:", len(cv_chunks), "| dropped tiny:", before - len(cv_chunks))
print("min/avg/max:", min(lens), sum(lens) // len(lens), max(lens))
print("math-heavy (<55% letters):", sum(a < 0.55 for a in alpha))
print("still hyphen-broken:", sum(len(re.findall(r"[a-z]- [a-z]", c["text"])) for c in cv_chunks))

out = "data/processed/cv_chunks.jsonl"
with open(out, "w", encoding="utf-8") as f:
    for i, c in enumerate(cv_chunks):
        f.write(json.dumps({"id": f"cv-{i:04d}", **c}, ensure_ascii=False) + "\n")
print(os.path.getsize(out) // 1024, "KB saved")

chunks: 800 | dropped tiny: 0
min/avg/max: 197 1121 1200
math-heavy (<55% letters): 4
still hyphen-broken: 13
1018 KB saved


In [14]:
chap_start = {2: 59, 3: 133, 5: 261, 6: 369, 7: 443}

# 1) check the chapter-opening pages we skip
for ch in KEEP:
    i = chap_start[ch]
    body = [l.strip() for l in sdoc[i - 1].get_text().split("\n") if l.strip() and ". . ." not in l]
    print("chapter", ch, "| non-TOC lines:", len(body), "|", " | ".join(body[:6])[:200])

# 2) drop math-heavy chunks and re-save
keep, dropped = [], []
for c in cv_chunks:
    a = sum(x.isalpha() for x in c["text"]) / len(c["text"])
    (dropped if a < 0.55 else keep).append(c)
for c in dropped:
    print("DROPPED:", c["section"], "| p.", c["pages"], "|", c["text"][:80])
cv_chunks = keep

with open("data/processed/cv_chunks.jsonl", "w", encoding="utf-8") as f:
    for i, c in enumerate(cv_chunks):
        f.write(json.dumps({"id": f"cv-{i:04d}", **c}, ensure_ascii=False) + "\n")
print("cv chunks:", len(cv_chunks), "| total all courses:", 510 + 307 + len(cv_chunks))

chapter 2 | non-TOC lines: 36 | Chapter 2 | Image formation | 2.1 | 36 | 2.1.1 | 40
chapter 3 | non-TOC lines: 69 | Chapter 3 | Image processing | 3.1 | 109 | 3.1.1 | 111
chapter 5 | non-TOC lines: 80 | Chapter 5 | Deep Learning | 5.1 | 239 | 5.1.1 | 241
chapter 6 | non-TOC lines: 30 | Chapter 6 | Recognition | 6.1 | 6.2 | 6.2.1 | 6.2.2
chapter 7 | non-TOC lines: 35 | Chapter 7 | Feature detection and matching | 7.1 | 7.1.1 | 7.1.2 | 7.1.3
DROPPED: 3.2 Linear filtering | p. 119-120 | 3.2 Linear filtering Locally adaptive histogram equalization is an example of a 
DROPPED: 3.3 More neighborhood operators | p. 131-132 | transformations. However non-linear filters, such as edge-preserving median or b
DROPPED: 3.3.3 Binary image processing | p. 139-140 | and blending (Section 8.4.2), and nearest point alignment (Section 13.2.1). The 
DROPPED: 3.4 Fourier transforms | p. 144-146 | (Gomes and Velho 1997) used as the “Gaussian” in Burt and Adelson’s (1983a) Lapl
cv chunks: 796 | total all cou

## 4. Convert Chunks to Different File Formats

To support multiple input formats, each course is exported in a different format. Every chunk keeps its `id`, `section`, `pages`, and `text`:
- **Python** → `.txt` (one `### CHUNK | ...` header per chunk)
- **MDL** → `.docx` (one Heading 2 per chunk, followed by its text)
- **CV** → `.csv` (one row per chunk)

Output folder: `data/raw_formats/`

In [15]:
os.makedirs("data/raw_formats", exist_ok=True)

with open("data/processed/python_chunks.jsonl", encoding="utf-8") as f:
    py = [json.loads(l) for l in f]

with open("data/raw_formats/python_tutorial.txt", "w", encoding="utf-8") as out:
    for c in py:
        out.write(f"### CHUNK | id={c['id']} | section={c.get('section') or ''} | pages={c.get('pages') or ''}\n")
        out.write(c["text"].strip() + "\n\n")

print("chunks written:", len(py))
print(os.path.getsize("data/raw_formats/python_tutorial.txt") // 1024, "KB")

chunks written: 307
323 KB


In [16]:
from docx import Document

with open("data/processed/mdl_chunks.jsonl", encoding="utf-8") as f:
    mdl = [json.loads(l) for l in f]

BAD = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

doc = Document()
doc.add_heading("Multimodal Deep Learning", level=1)

for c in mdl:
    doc.add_heading(
        f"CHUNK | id={c['id']} | section={c.get('section') or ''} | pages={c.get('pages') or ''}",
        level=2,
    )
    doc.add_paragraph(BAD.sub("", c["text"].strip()))

doc.save("data/raw_formats/mdl_course.docx")
print("chunks written:", len(mdl))

chunks written: 510


In [17]:
import csv

with open("data/processed/cv_chunks.jsonl", encoding="utf-8") as f:
    cv = [json.loads(l) for l in f]

with open("data/raw_formats/cv_course.csv", "w", encoding="utf-8-sig", newline="") as out:
    w = csv.writer(out)
    w.writerow(["id", "section", "pages", "text"])
    for c in cv:
        w.writerow([c["id"], c.get("section") or "", c.get("pages") or "", c["text"].strip()])

print("rows written:", len(cv))

rows written: 796


## 5. Loaders and Round-Trip Test

One loader per format (`.txt`, `.docx`, `.csv`) returns chunks in the same schema. The test confirms that each file loads back with the same number of chunks and the same ids as the original `.jsonl`.

The MDL text differs slightly in 10 chunks: the `.docx` format cannot store control characters, which the PDF extraction had left in the formula text. They carry no meaning, so removing them is harmless.

In [18]:
COURSES = {
    "python": ("Python 3 Tutorial", "py"),
    "mdl": ("Multimodal Deep Learning", "mdl"),
    "cv": ("Computer Vision (Szeliski)", "cv"),
}
HEAD = re.compile(r"CHUNK \| id=(.*?) \| section=(.*?) \| pages=(.*)$")

def _mk(cid, course, source, section, pages, text):
    return {"id": cid, "course": course, "source": source,
            "section": section or "", "pages": pages or "", "text": text.strip()}

def load_txt(path, course):
    chunks, cur, buf = [], None, []
    def flush():
        if cur:
            chunks.append(_mk(cur[0], course, os.path.basename(path), cur[1], cur[2], "\n".join(buf)))
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("### CHUNK | "):
                flush()
                cur = HEAD.match(line[len("### "):]).groups()
                buf = []
            else:
                buf.append(line)
    flush()
    return chunks

def load_docx(path, course):
    chunks, cur, buf = [], None, []
    def flush():
        if cur:
            chunks.append(_mk(cur[0], course, os.path.basename(path), cur[1], cur[2], "\n".join(buf)))
    for p in Document(path).paragraphs:
        if p.style.name == "Heading 2" and p.text.startswith("CHUNK | "):
            flush()
            cur = HEAD.match(p.text).groups()
            buf = []
        elif cur is not None:
            buf.append(p.text)
    flush()
    return chunks

def load_csv(path, course):
    with open(path, encoding="utf-8-sig", newline="") as f:
        return [_mk(r["id"], course, os.path.basename(path), r["section"], r["pages"], r["text"])
                for r in csv.DictReader(f)]

LOADERS = {".txt": load_txt, ".docx": load_docx, ".csv": load_csv}

def load_file(path, course):
    return LOADERS[os.path.splitext(path)[1].lower()](path, course)

# test the round-trip
files = {
    "python": "data/raw_formats/python_tutorial.txt",
    "mdl": "data/raw_formats/mdl_course.docx",
    "cv": "data/raw_formats/cv_course.csv",
}
for key, path in files.items():
    new = load_file(path, COURSES[key][0])
    with open(f"data/processed/{key}_chunks.jsonl", encoding="utf-8") as f:
        old = [json.loads(l) for l in f]
    same_ids = [c["id"] for c in new] == [c["id"] for c in old]
    same_text = sum(a["text"].strip() == b["text"].strip() for a, b in zip(new, old))
    print(f"{key}: loaded={len(new)} original={len(old)} | ids match={same_ids} | text match={same_text}/{len(old)}")

python: loaded=307 original=307 | ids match=True | text match=307/307
mdl: loaded=510 original=510 | ids match=True | text match=500/510
cv: loaded=796 original=796 | ids match=True | text match=796/796
